Exercises:
- E01: I did not get around to seeing what happens when you initialize all weights and biases to zero. Try this and train the neural net. You might think either that 1) the network trains just fine or 2) the network doesn't train at all, but actually it is 3) the network trains but only partially, and achieves a pretty bad final performance. Inspect the gradients and activations to figure out what is happening and why the network is only partially training, and what part is being trained exactly.
- **E02: BatchNorm, unlike other normalization layers like LayerNorm/GroupNorm etc. has the big advantage that after training, the batchnorm gamma/beta can be "folded into" the weights of the preceeding Linear layers, effectively erasing the need to forward it at test time. Set up a small 3-layer MLP with batchnorms, train the network, then "fold" the batchnorm gamma/beta into the preceeding Linear layer's W,b by creating a new W2, b2 and erasing the batch norm. Verify that this gives the same forward pass during inference. i.e. we see that the batchnorm is there just for stabilizing the training, and can be thrown out after training is done! pretty cool.**

In [3]:
import ctypes
from pathlib import Path

# On this host, load the native NVIDIA driver before PyTorch's CUDA compatibility library.
native_driver = Path("/usr/lib/x86_64-linux-gnu/libcuda.so.1")
if native_driver.is_file():
  ctypes.CDLL(str(native_driver), mode=ctypes.RTLD_GLOBAL)

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

device = torch.device("cuda")
try:
  gpu_name = torch.cuda.get_device_name(device)
except RuntimeError as error:
  if "Error 804" in str(error):
    raise RuntimeError("This kernel already loaded the CUDA compatibility driver. Click Restart Kernel in the notebook toolbar, then run this first cell. The NN (CUDA) Jupyter kernel also loads the system NVIDIA driver at startup.") from None
  raise
print(f"Training on {device}: {gpu_name}")

Training on cuda: NVIDIA GeForce RTX 3090


In [4]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [5]:
len(words)

32033

In [6]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [7]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X, device=device)
  Y = torch.tensor(Y, device=device)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%


torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [8]:
# Let's train a deeper network
# The classes we create here are the same API as nn.Module in PyTorch

class Linear:
  
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out), generator=train_generator, device=device) / fan_in**0.5
    # self.weight = torch.zeros((fan_in, fan_out), device=device)
    self.bias = torch.zeros(fan_out, device=device) if bias else None
  
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim, device=device)
    self.beta = torch.zeros(dim, device=device)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim, device=device)
    self.running_var = torch.ones(dim, device=device)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # Update the same buffers across CUDA Graph replays.
    if self.training:
      with torch.no_grad():
        self.running_mean.lerp_(xmean.squeeze(0), self.momentum)
        self.running_var.lerp_(xvar.squeeze(0), self.momentum)
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
train_generator = torch.Generator(device=device).manual_seed(2147483647) # for reproducibility

C = torch.randn((vocab_size, n_embd), generator=train_generator, device=device)
layers = [
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]

with torch.no_grad():
  # last layer: make less confident
  layers[-1].gamma *= 0.1
  # layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 1.0 #5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

16424


In [10]:
# SGD training
max_steps = 200000
batch_size = 32


def train_model(max_steps, batch_size):
  loss_history = torch.empty(max_steps, device=device)
  update_history = torch.empty((max_steps, len(parameters)), device=device)
  step_index = torch.zeros(1, dtype=torch.long, device=device)

  for layer in layers:
    layer.training = True

  def train_step():
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=train_generator, device=device)
    emb = C[Xtr[ix]]
    x = emb.flatten(1)
    for layer in layers:
      x = layer(x)
    loss = F.cross_entropy(x, Ytr[ix])

    for layer in layers:
      layer.out.retain_grad()
    for p in parameters:
      p.grad = None
    loss.backward()

    with torch.no_grad():
      lr = torch.where(step_index < 150000, 0.1, 0.01)
      for p in parameters:
        p.addcmul_(p.grad, lr, value=-1.0)

      ratios = torch.stack([p.grad.std() / p.std() for p in parameters])
      loss_history.index_copy_(0, step_index, loss.log10().reshape(1))
      update_history.index_copy_(0, step_index, (lr * ratios).log10().unsqueeze(0))
      step_index.add_(1)

  # Warm up the CUDA kernels on a side stream, counting these as training steps.
  warmup_steps = min(3, max_steps)
  warmup_stream = torch.cuda.Stream(device=device)
  current_stream = torch.cuda.current_stream(device)
  warmup_stream.wait_stream(current_stream)
  with torch.cuda.stream(warmup_stream):
    for _ in range(warmup_steps):
      train_step()
  current_stream.wait_stream(warmup_stream)

  if max_steps:
    print(f'{0:7d}/{max_steps:7d}: {10.0 ** loss_history[0].item():.4f}')
  if max_steps > warmup_steps:
    # Capture one fixed-size minibatch step to reuse its CUDA kernel launches.
    graph = torch.cuda.CUDAGraph()
    graph.register_generator_state(train_generator)
    with torch.cuda.graph(graph):
      train_step()
    for i in range(warmup_steps, max_steps):
      graph.replay()
      if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {10.0 ** loss_history[i].item():.4f}')

  return loss_history.cpu(), update_history.cpu()


lossi, ud = train_model(max_steps, batch_size)

# Subsequent evaluation and sampling overwrite layer.out.
tanh_diagnostics = [
  (i, layer.out.detach().cpu(), layer.out.grad.detach().cpu())
  for i, layer in enumerate(layers)
  if isinstance(layer, Tanh)
]

      0/ 200000: 2.2575
  10000/ 200000: 2.2559
  20000/ 200000: 2.2199
  30000/ 200000: 2.3295
  40000/ 200000: 2.1494
  50000/ 200000: 1.9479
  60000/ 200000: 2.6247
  70000/ 200000: 2.0338
  80000/ 200000: 2.5969
  90000/ 200000: 1.8201
 100000/ 200000: 1.8851
 110000/ 200000: 2.1226
 120000/ 200000: 1.9487
 130000/ 200000: 2.1647
 140000/ 200000: 2.1058
 150000/ 200000: 2.0728
 160000/ 200000: 2.4381
 170000/ 200000: 1.9941
 180000/ 200000: 2.3389
 190000/ 200000: 2.3484


In [11]:
from copy import copy

@torch.no_grad()
def fold_linear_bn(linear, bn):
    scale = bn.gamma / torch.sqrt(bn.running_var + bn.eps)

    folded = copy(linear)
    folded.weight = linear.weight * scale
    folded.bias = bn.beta - bn.running_mean * scale
    return folded

linear1, bn1, _, linear2, bn2, _, linear3, bn3 = layers

# 原模型的 BN 切到推理模式，使用固定的 running stats。
for bn in (bn1, bn2, bn3):
    bn.training = False

folded_layers = [
    fold_linear_bn(linear1, bn1), Tanh(),
    fold_linear_bn(linear2, bn2), Tanh(),
    fold_linear_bn(linear3, bn3),
]

@torch.no_grad()
def inference_logits(network, contexts):
    x = C[contexts].flatten(1)
    for layer in network:
        x = layer(x)
    return x

contexts = Xdev[:256]
before = inference_logits(layers, contexts)
after = inference_logits(folded_layers, contexts)

print("最大绝对误差", (before - after).abs().max().item())
assert torch.allclose(before, after, rtol=1e-5, atol=1e-5)

最大绝对误差 3.635883331298828e-06


In [12]:
@torch.no_grad()
def dataset_loss(network, inputs, targets, batch_size=4096):
    total = torch.zeros((), device=inputs.device, dtype=torch.float64)

    for xb, yb in zip(inputs.split(batch_size), targets.split(batch_size)):
        logits = inference_logits(network, xb)
        losses = F.cross_entropy(logits, yb, reduction="none")
        total += losses.sum(dtype=torch.float64)

    return (total / len(targets)).item()


for name, inputs, targets in [
    ("train", Xtr, Ytr),
    ("val", Xdev, Ydev),
]:
    original_loss = dataset_loss(layers, inputs, targets)
    folded_loss = dataset_loss(folded_layers, inputs, targets)

    print(f"{name} | 原版 {original_loss:.8f} | 折叠版 {folded_loss:.8f}"
          f" | 差值 {abs(original_loss - folded_loss):.2e}")

train | 原版 2.04146331 | 折叠版 2.04146333 | 差值 1.54e-08
val | 原版 2.09901647 | 折叠版 2.09901649 | 差值 1.42e-08


训练结束后，可以把 BatchNorm 的缩放和平移合进前一层 Linear，用折叠后的 Linear 完成同样的推理计算。实验中，256 条验证样本的 logits 最大绝对误差为 2.62e-6，完整训练集和验证集的 loss 差分别为 2.48e-8 和 2.30e-8。折叠前后的结果在浮点精度范围内一致。